In [11]:
import numpy as np
import h5py
import xarray as xr

def load_ep_flux_data(pr, save_dir):
    f = np.load(f"{save_dir}/epflux_PR{pr}.npz", "r")
    Fj = f["ep1"][:]  # (time, level, lat)
    Fk = f["ep2"][:]
    lat = f["lat"][:]
    pres = f["pres"][:]
    return Fj, Fk, lat, pres

def composite_mean(F, time_idx_array, lags):
    samples = []
    for lag in lags:
        time_idx_array_lag = time_idx_array + lag
        time_idx_array_lag = time_idx_array_lag[time_idx_array_lag < F.shape[0]]
        samples.append(F[time_idx_array_lag].mean(axis=0))
    return np.mean(samples, axis=0)

# Configuration
save_stat_dir = "/data92/PeterChang/back_to_master1220/Moist_Dycore/"
save_dir = "/data92/PeterChang/back_to_master1220/Moist_Dycore/EPflux"

pr_list = [0, 10, 20, 30, 40, 50]
# pr_list = [0]

lags_5_20 = np.arange(5*4, 25*4+1, 1)
n_samples = 500

np.random.seed(0)  # Fix random seed for reproducibility

for PR in pr_list:
    print(f"\nLoading EP flux data for PR={PR}...")
    
    Fj, Fk, latitude, pres = load_ep_flux_data(PR, save_dir)
    Ntime = Fj.shape[0]
    print(Ntime)
    print(f"Loading time indices for PR={PR}...")
    with h5py.File(f"{save_dir}/New_PC1_time_idx/PC1_positive_1std_PR{PR}.h5") as f:
        time_idx_pos = f["time_idx_pos"][:]
    with h5py.File(f"{save_dir}/New_PC1_time_idx/PC1_negative_1std_PR{PR}.h5") as f:
        time_idx_neg = f["time_idx_neg"][:]

    n_pos = len(time_idx_pos[0])
    n_neg = len(time_idx_neg[0])

    print("--> Performing Monte Carlo sampling...")
    diff_phi_samples = np.empty((n_samples, Fj.shape[1], Fj.shape[2]), dtype=np.float32)
    diff_p_samples   = np.empty((n_samples, Fk.shape[1], Fk.shape[2]), dtype=np.float32)

    for i in range(n_samples):
        rand_idx_pos = np.random.choice(Ntime, size=n_pos, replace=False)
        rand_idx_neg = np.random.choice(Ntime, size=n_neg, replace=False)
        print(rand_idx_pos.shape, f"Now running {i}th time for PR{PR}...")
        
        rand_ep1_pos = composite_mean(Fj, rand_idx_pos, lags_5_20)
        rand_ep2_pos = composite_mean(Fk, rand_idx_pos, lags_5_20)
        rand_ep1_neg = composite_mean(Fj, rand_idx_neg, lags_5_20)
        rand_ep2_neg = composite_mean(Fk, rand_idx_neg, lags_5_20)

        diff_phi_samples[i] = rand_ep1_pos - rand_ep1_neg
        diff_p_samples[i]   = rand_ep2_pos - rand_ep2_neg

    print("--> Saving Monte Carlo samples...")
    ds_samples = xr.Dataset({
        f"EP_phi_diff_rand_PR{PR}": (["sample", "level", "lat"], diff_phi_samples),
        f"EP_p_diff_rand_PR{PR}":   (["sample", "level", "lat"], diff_p_samples),
    }, coords={"sample": np.arange(n_samples), "level": pres, "lat": latitude})

    save_path_samples = f"{save_stat_dir}/EPflux/statistical_test_for_EP_diff_PR{PR}.nc"
    ds_samples.to_netcdf(save_path_samples)
    print("✓ Monte Carlo save complete.")




Loading EP flux data for PR=0...
78000
Loading time indices for PR=0...
--> Performing Monte Carlo sampling...
(12482,)
(12482,)
(12482,)
(12482,)
(12482,)
(12482,)


KeyboardInterrupt: 